In [97]:
import numpy as np


from qiskit import QuantumCircuit,  QuantumRegister, ClassicalRegister, AncillaRegister, transpile, qasm2
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.circuit.library import PauliEvolutionGate, hamiltonian_variational_ansatz
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.synthesis import SuzukiTrotter
from qiskit.primitives import BackendEstimatorV2 as Estimator

import os

import quimb.tensor as qtn
import quimb as qu

import csv
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.ticker as tck

In this Jupyter notebook, I use the quimb tensor network package to approximate long TFIM chains, and then use the properties of MPS to evolve and measure the state in order to calculate the dichotomic time correlator $C_Q(t) = \langle \psi |\frac{1}{2} \{e^{i H t} Q e^{ - i H t} Q \} | \psi \rangle $

In [ ]:
def TFIM_DMRG_correlation(L, J = 1, h = 1, time_range = np.linspace(0,2,.1), measure_site = None, measure_opp = 'X'):
    """Calcuates C_Q(t) for t in time_range and outputs an array of C_Q(t)"""

    #declare Hamiltonian. Default J is antiferromagnetic 
    # negative Hamiltonian is for evolving backwards in time
    H_MPO = qtn.MPO_ham_ising(L, j=-1.0 * J, bx=h, cyclic= False)
    H_MPO_neg = qtn.MPO_ham_ising(L, j= J, bx=-1.0 * h, cyclic= False)

    #run the DMRG procedure
    dmrg = qtn.DMRG2(H_MPO)
    dmrg.solve(max_sweeps=20, verbosity=0, cutoffs = 1e-09 )

    #call the ground state as a tensor network
    psi0 =dmrg.state


In [140]:
L = 20
J = .1
h = 1

H_MPO = qtn.MPO_ham_ising(L, j=-1.0 * J, bx=h, cyclic= False)

dmrg = qtn.DMRG2(H_MPO)

dmrg.solve(
    max_sweeps=20,
    verbosity=0,
    cutoffs = 1e-09
)

psi0 =dmrg.state

In [104]:
gsh = gs.H
gs.align_(H_MPO, gsh)
(gsh & H_MPO & gs) ^ ...

-20.297927178737584

In [147]:
H = qtn.ham_1d_ising(L, j=-1.0 * J, bx=h, cyclic= False)
H_neg = qtn.ham_1d_ising(L, j= J, bx= -1.0 * h, cyclic= False)

tebd = qtn.TEBD(
    psi0,
    H,
    dt=0.05
)

tebd.update_to(T =   1.0, progbar = False)
#psi_t = tebd.pt


In [148]:
X = np.array([[0, 1], [1, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

    
# Copy the state
psi_after = psi0.copy()
    
    # Apply X gate to site n
psi_after = psi_after.gate(X, 11, contract='swap+split')


psi_after.H @ psi0

np.complex128(0.99937470672396+0j)

In [107]:
X = qtn.Tensor(qu.pauli("X"), inds=("k0", "b0"), tags=["PAULI", "X", "0"])

In [121]:
qu.spin_operator('X')

[[0. +0.j 0.5+0.j]
 [0.5+0.j 0. +0.j]]

In [144]:
qtn.MPO_ham_ising(L, j=-1.0 * J, bx=h, cyclic= False)

MatrixProductOperator(tensors=20, indices=59, L=20, max_bond=3)

In [151]:
H_loc  = qtn.ham_1d_ising(L, j=-1.0 * J, bx=h, cyclic= False)
H_loc.build_mpo()

AttributeError: 'LocalHam1D' object has no attribute 'build_mpo'

In [154]:
import quimb.operator as qop

hilbert_space = qop.HilbertSpace(
    sites=L,
    symmetry="U1",
    sector=L//2,
)
H = qop.SparseOperatorBuilder(hilbert_space=hilbert_space)
H.buid_mpo()


AttributeError: 'SparseOperatorBuilder' object has no attribute 'buid_mpo'

In [157]:
builder = qtn.SpinHam1D(S=1)
builder += 1 / 2, "+", "-"
builder += 1 / 2, "-", "+"
builder += 1, "Z", "Z"
H = builder.build_mpo(L=100)
builder